# 25 - Full Carvana inventory: collection and coverage

**Question:** Did the expanded collector finish its declared inventory, and where are the gaps?

**Inputs:** retained full-inventory reports and an optional reviewed export. **Outputs:** read-only coverage tables. Run All never collects or writes. This workflow is separate from the frozen 101-query experiment. Read `vehicle/docs/full_inventory.md` for operation and `full_inventory_goal.md` for acceptance criteria.

A VIN identifies a vehicle. A query selects a make or make/model family at a ZIP. An observation is a vehicle on a retained search page at its actual observation time. A six-hour sweep is not a simultaneous snapshot. Missing vehicles are not automatically sales.


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
REPO = next(p for p in [here, *here.parents] if (p / 'vehicle/config/carvana_full_inventory.json').is_file())
config = json.loads((REPO / 'vehicle/config/carvana_full_inventory.json').read_text())
CAPTURE_ROOT = REPO / 'vehicle' / config['capture_root']
EXPORT = None  # Optional: Path to one existing replay/export folder.
print('Request ceiling:', config['max_requests'], '| Maximum hours:', config['max_seconds'] / 3600)
print('Primary ZIP:', config['primary_zip'], '| Validation ZIPs:', config['validation_zips'])


## Collection register

These are operational summaries, not verified sales or proof of national coverage. `primary_queries_complete` means all selected primary queries finished. `primary_scope_reconciled` additionally requires no duplicate primary membership and zero opening/closing count residual. Geographic checks cover a limited rotating sample. Missing dates are not zero-inventory dates.


In [ ]:
records = []
for path in sorted(CAPTURE_ROOT.glob('*/catalog_report.json')):
    report = json.loads(path.read_text(encoding='utf-8'))
    records.append({key: report.get(key) for key in [
        'cycle_date', 'status', 'requests', 'discovery_complete', 'primary_queries_complete',
        'primary_scope_reconciled', 'primary_observed_vins', 'opening_count_residual',
        'closing_count_residual', 'geographic_membership_stable', 'started_at', 'ended_at']})
collection_register = pd.DataFrame(records)
if collection_register.empty:
    print('No full-inventory baseline retained yet. Updated code and budget are not collected evidence.')
else:
    display(collection_register)


## Inspect one replayed export

Set `EXPORT` to an existing export when ready. Its manifest binds source and output bytes. Discovery samples and geographic checks are excluded from primary inventory, except complete make probes. A blocked replay requires evidence review, not replacement with a fresh attempt.


In [ ]:
if EXPORT is not None:
    EXPORT = Path(EXPORT)
    manifest = json.loads((EXPORT / 'manifest.json').read_text(encoding='utf-8'))
    for path, expected in manifest['sources'].items():
        assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected, f'Changed source: {path}'
    for name, expected in manifest['outputs'].items():
        assert hashlib.sha256((EXPORT / name).read_bytes()).hexdigest() == expected, f'Changed export: {name}'
    coverage = pd.read_csv(EXPORT / 'coverage.csv')
    display(coverage)
    display(coverage.loc[~coverage['query_complete'].fillna(False)])
    display(pd.read_json(EXPORT / 'geographic_checks.json'))
else:
    print('No export selected; nothing has been imported, refreshed or collected.')


## Move from coverage to sales research

Compare each VIN with all retained positive observations, preserving first/last observed times and source scope. Only comparable complete coverage can support absence. New category coverage can discover old inventory: first observed does not mean newly listed.

Continue to Notebook 20 for compatible inventory/price comparisons and Notebook 24 for native-status validation. Keep pending, absent, native Sold and transaction-confirmed sale separate. The existing small validation study cannot establish a companywide sales conversion rate.
